<a href="https://colab.research.google.com/github/subhadeepm465/data_engineering_trng/blob/main/Pyspark_Session1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("PysparkPractice").getOrCreate()

In [0]:
# Install the faker library
%pip install faker

from pyspark.sql.types import *
import random
from faker import Faker

# Initialize Faker
fake = Faker()

# Generate synthetic user data
def generate_user_data(n=1000):
    genders = ['M', 'F']
    cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix']
    data = []

    for user_id in range(1, n + 1):
        name = fake.first_name()
        age = random.randint(18, 60)
        gender = random.choice(genders)
        city = random.choice(cities)
        income = random.randint(30000, 120000)
        data.append((user_id, name, age, gender, city, income))

    return data

# Define schema
schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("city", StringType(), True),
    StructField("income", IntegerType(), True)
])

# Create DataFrame
user_data = generate_user_data(1000)
users = spark.createDataFrame(user_data, schema)

# Preview Data
display(users)
users.printSchema()


In [0]:
from pyspark.sql.functions import col
users1=users.filter(col("income") > 60000)
users1.show(5)

In [0]:
users.show()

30+ Basic PySpark Practice Questions
🧱 Data Exploration
Show the first 5 rows of the DataFrame.

Print the schema of the DataFrame.

Count the number of rows in the DataFrame.

List all column names and data types.

Show summary statistics (describe()).

🔍 Filtering & Conditional Logic
Filter users with income > 60000.

Filter female users (gender == 'F').

Filter users who live in Chicago and are under 30.

Filter users whose name starts with 'A'.

Create a new column: is_high_income (True if income > 60000).

📊 Aggregations
Find the average income.

Find the maximum and minimum age.

Count users by gender.

Count users by city.

Find total income by city.

🧮 GroupBy and Aggregations
Find average income by gender.

Find the number of users by age group (use bucketing).

Show count and average income by city and gender.

🔁 Sorting and Ranking
Sort users by income descending.

Sort users by age ascending and then by income descending.

✂️ Column Operations
Add a new column age_plus_ten = age + 10.

Rename income column to annual_income.

Drop the city column.

Reorder columns: user_id, name, gender, age, income.

🔗 Joins (create another small DF for joins)
python
Copy
Edit
cities = spark.createDataFrame([
    ("New York", "NY"),
    ("Los Angeles", "CA"),
    ("Chicago", "IL")
], ["city", "state"])
Join users with the cities DataFrame on city.

🧹 Data Cleaning
Replace null incomes with the average income.

Filter out rows where name is null.

Detect and remove duplicates.

🧠 Window Functions
Add a rank to users by income (highest to lowest).

Add a column showing average income per city using a window.

Calculate the running total of income ordered by age.

💾 Data I/O (if writing to Delta/Parquet)
Save the DataFrame as a Parquet file.

Read it back and show it.

Save the DataFrame as a Delta table (Databricks only).


🧱 **Data Exploration**

In [0]:
users.show(5)
users.printSchema()
print(users.count())
print(users.dtypes)
users.describe().show()
users.columns



In [0]:
users.select("user_id","name").show(5)

In [0]:
catalog_name = "workspace"   # Replace with your actual catalog name
schema_name = "default"
table_name="users1"

In [0]:
users1.write.mode("overwrite").saveAsTable(f"{catalog_name}.{schema_name}.{table_name}")

In [0]:
from pyspark.sql.functions import *

**🔍 Filtering & Conditional Logic**

In [0]:
#users.filter(col("income") > 60000).show(5)
#users.where(col("income") > 60000).select('name','income').show(5)

#users.filter(col("gender") == 'F').show()
#users.filter((col("city") == "Chicago") | (col("age") < 30)).show(5)

#users.filter((col("city") == "Chicago") & (col("age") < 30)).show(5)
#users.filter(col("name").startswith("A")).show(5)
users = users.withColumn("is_high_income", col("income") > 60000)
users.show(5)
#withColumn('col_name',expression)


**📊 Aggregations**

In [0]:

#users.select(avg("income")).show(5)
#users.select(max("age"), min("age")).show()
#users.groupBy("gender").count().show()
#users.groupBy("city").count().show()
#users.groupBy("city").agg({"income": "sum"}).withColumnRenamed('sum(income)','total_income').show(5)
#users.groupBy("city").sum("income").withColumnRenamed("sum(income)", "total_income").show()
#users.groupBy("city").sum("income").withColumnRenamed("sum(income)", "total_income").show()



**Using sparkSQL**

In [0]:
#For using the data in spark sql the dataframe needs to have a temp view. Users is the temp view below
users.createOrReplaceTempView("users")
spark.sql("SELECT AVG(income) AS avg_income FROM users").show()
spark.sql("SELECT MAX(age) AS max_age, MIN(age) AS min_age FROM users").show()
spark.sql("SELECT gender, COUNT(*) AS user_count FROM users GROUP BY gender").show()
spark.sql("SELECT city, COUNT(*) AS user_count FROM users GROUP BY city").show()
spark.sql("SELECT city, SUM(income) AS total_income FROM users GROUP BY city").show()


**🧮 GroupBy and Aggregations**

In [0]:
users1 = users.withColumn("age_grp",when(col('age')<=20,'0-20').when(col('age')<=30,'0-30').when(col('age')<=40,'0-40').otherwise('40+')).drop('age_group').show(5)

In [0]:
#users.groupBy("gender").agg(avg("income")).show()

# Age bucketing (e.g., 0–20, 21–30, etc.)
users = users.withColumn("age_group",
    when(col("age") <= 20, "0-20")
    .when(col("age") <= 30, "21-30")
    .otherwise("30+"))
users.groupBy("age_group").count().show()

users.groupBy("city", "gender")\
     .agg(count("*").alias("user_count"), avg("income").alias("avg_income")).show()


In [0]:
users.groupBy("city").sum("income").withColumnRenamed("sum(income)", "total_income").show()

In [0]:
users.groupBy("city").sum("income").withColumnRenamed("sum(income)", "total_income").orderBy(col('total_income').desc()).show()

**🔁 Sorting and Ranking**

In [0]:
#users.orderBy(desc("income")).show(5)
users.orderBy(("age"), desc("income")).show(5)
users.orderBy(("age"), col('income').desc()).show(5)


**✂️ Column Operations**

In [0]:
users = users.withColumn("age_plus_ten", col("age") + 10)
users = users.withColumnRenamed("income", "annual_income")
users = users.drop("city")
users = users.select("user_id", "name", "gender", "age", "annual_income")


**🔗 Joins**

In [0]:
cities = spark.createDataFrame([
    ("New York", "NY"),
    ("Los Angeles", "CA"),
    ("Chicago", "IL")
], ["city", "state"])

# Need to bring 'city' back before joining
users = users.withColumnRenamed("annual_income", "income")  # revert for joining
users = users.withColumn("city",
    when(col("user_id") == 1, "Chicago")
    .when(col("user_id") == 2, "New York")
    .when(col("user_id") == 3, "Los Angeles")
    .when(col("user_id") == 4, "Chicago")
    .when(col("user_id") == 5, "New York"))

users.join(cities, on="city", how="left").show()


**🧹 Data Cleaning**

In [0]:
avg_income_val = users.select(avg("income")).first()[0]
users = users.fillna({"income": avg_income_val})
users = users.filter(col("name").isNotNull())
users = users.dropDuplicates()


**💾 Data I/O**

In [0]:
df = spark.read.csv("/content/drive/MyDrive/data.csv",header=True)
df.show(5)

#data abc;
#infile '/path/data.csv' dlm = ','
#run;

In [0]:
df = spark.read.format('csv').option('header','true').option('inferSchema','true').load('/content/drive/MyDrive/data.csv')
df.show(5)

In [0]:
# Save as Parquet
users.write.mode("overwrite").parquet("/content/drive/MyDrive/")
# Read it back
parquet_df = spark.read.parquet("/tmp/users_parquet")
parquet_df.show()

# Save as Delta (Databricks only)
users.write.format("delta").mode("overwrite").save("/tmp/users_delta")
# Read back
delta_df = spark.read.format("delta").load("/tmp/users_delta")
delta_df.show()
